# Parameter sweep: alpha, beta_min, c (growth burden, private benefit, inhibitor efficacy)

**Assignment.** This notebook covers one person's slice of the group's four-way parameter
sweep on the Ma et al. (2024, *Nat Commun* 15:8337) sensitive/resistant + beta-lactamase
model: **alpha** (growth-burden factor), **beta_min** (baseline private-benefit factor), and
**c** (inhibitor's intracellular efficacy). These three are grouped together because they are
exactly the variables behind the paper's own selection criterion (their Eqs. 1-2), so this
sweep doubles as a criterion-validation exercise (their Fig. 3A/3C, done here as a dense
structured grid instead of the paper's 10,000 randomized strains).

**Parameters now match the paper's Supplementary Information exactly.**

| Parameter | SI base value | Swept here? | Range used here (= SI randomized range) |
|---|---|---|---|
| alpha | 0.95 | yes | 0.75 - 1 |
| beta_min | 0.9 | yes | 0 - 1 |
| c | 0.7 | yes | 0 - 1 |
| gamma | 1.38 | no (held) | SI range 1.1 - 1.4 |
| h_a | 3 | no (held) | SI range 1 - 5 |
| h_i | 2 | no (held) | SI range 1 - 5 |
| kappa_b | 0.35 | no (held) | SI range 0 - 1 |
| d_a | 0.02 | no (held) | not randomized in SI |
| d_b | 1 | no (held) | SI range 1 - 10 |
| xi | 0.8 | no (held) | SI range 0 - 1 |
| phi_max | 1 | no (held) | SI range 0 - 5 |

Initial conditions (SI Section 1): n_s(0) = n_r(0) = 0.2, s(0) = 4, b(0) = 0; antibiotic
dose 1 < a0 < 100 and inhibitor dose 0.1 <= i <= 10.

**Requested endpoints**, computed at every grid point:
- community diversity (Shannon index, final and minimum-during-treatment)
- time to reach steady state
- final resistant fraction f_R = n_r / (n_s + n_r)
- whether R is driven below an extinction threshold
- whether S survives and recovers to a viable density
- time-to-recovery for S
- peak population loss during treatment (bottleneck depth)

**Two dosing scenarios** are swept, for different purposes:
1. **`treatment`** (a0 = 2.15, i = 10) -- the single dose of the paper's Fig. 2C. With the
   SI base parameters this reproduces the published time course (both populations dip,
   then regrow to S ~ 2.2, R ~ 1.7, with f_R falling from 0.5 to ~0.43). All the
   endpoints above are computed here.
2. **`criterion`** (a0 = 100, i = 10) -- the highest antibiotic and inhibitor doses in the
   paper's dose range, i.e. the closest the paper's own dose grid gets to the saturating
   limit its simplified criterion (Eq. 2) assumes. Used only to validate that criterion
   (notebook 3).

**What this notebook does:** (1) verifies the sweep engine's log-space reformulation
against the already-reviewed `ma2024_model.py`, (2) sanity-checks the integration horizon
across the parameter cube's corners and a random sample, (3) runs both grids in parallel
and saves the results to CSV for notebooks 2 and 3.

**Held fixed:** gamma, h_a, h_i, kappa_b, d_a, d_b, xi, phi_max stay at their SI base values
(`ma2024_model.DEFAULT_PARAMS`) -- those are other group members' slices of the sweep.

## Setup

In [1]:
import time
import numpy as np
import pandas as pd

from ma2024_model import DEFAULT_PARAMS, SI_RANDOMIZED_RANGES, SI_INITIAL_CONDITIONS
from sweep_engine import (
    verify_equivalence, compute_endpoints, criterion_check, no_drug_reference,
    EXTINCTION_THRESHOLD, VIABLE_FRACTION, RECOVERY_FRACTION, TOL_SS,
)
from grid_runner import run_treatment_grid, run_criterion_grid

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
print("Base parameters (SI Section 2):", DEFAULT_PARAMS)
print("Initial conditions (SI Section 1):", SI_INITIAL_CONDITIONS)
ALPHA_RANGE = SI_RANDOMIZED_RANGES["alpha"]
BETA_MIN_RANGE = SI_RANDOMIZED_RANGES["beta_min"]
C_RANGE = SI_RANDOMIZED_RANGES["c"]
print(f"Swept ranges (SI randomized ranges): alpha {ALPHA_RANGE}, beta_min {BETA_MIN_RANGE}, c {C_RANGE}")
print()
print("Thresholds in force for every endpoint below:")
print(f"  EXTINCTION_THRESHOLD = {EXTINCTION_THRESHOLD}  (absolute density)")
print(f"  VIABLE_FRACTION      = {VIABLE_FRACTION}  (of the drug-free control's steady n_s)")
print(f"  RECOVERY_FRACTION    = {RECOVERY_FRACTION}  (of S's own eventual steady-state value)")
print(f"  TOL_SS               = {TOL_SS}  (per-capita growth rate, 1/time)")


Base parameters (SI Section 2): Params(alpha=0.95, beta_min=0.9, c=0.7, gamma=1.38, h_a=3.0, h_i=2.0, kappa_b=0.35, d_a=0.02, d_b=1.0, xi=0.8, phi_max=1.0)
Initial conditions (SI Section 1): {'ns0': 0.2, 'nr0': 0.2, 's0': 4.0, 'b0': 0.0}
Swept ranges (SI randomized ranges): alpha (0.75, 1.0), beta_min (0.0, 1.0), c (0.0, 1.0)

Thresholds in force for every endpoint below:
  EXTINCTION_THRESHOLD = 0.0001  (absolute density)
  VIABLE_FRACTION      = 0.5  (of the drug-free control's steady n_s)
  RECOVERY_FRACTION    = 0.9  (of S's own eventual steady-state value)
  TOL_SS               = 0.001  (per-capita growth rate, 1/time)


## Step 1 -- Verify the sweep engine against the reviewed model

`sweep_engine.py` re-derives the same five ODEs in log-space (tracking ln n_s, ln n_r
instead of n_s, n_r) so that populations can be integrated through a deep bottleneck
without losing precision. Before trusting it for anything, we check it reproduces
`ma2024_model.simulate()` (the already-delivered, already-reviewed implementation)
at randomized parameter points, in both dosing regimes.

In [2]:
diffs_treatment = verify_equivalence(n_checks=30, seed=1, a0=2.15, i_dose=10.0, t_span=(0.0, 40.0))
print("Treatment-scenario equivalence check (worst excess beyond atol+rtol*|raw|, as a")
print("fraction of the allowed tolerance; <=0 means within tolerance):")
for k, v in diffs_treatment.items():
    print(f"  {k}: {v:.4f}")

diffs_criterion = verify_equivalence(n_checks=20, seed=2, a0=100.0, i_dose=10.0, t_span=(0.0, 80.0))
print()
print("Criterion-scenario (a0=100, i=10) equivalence check:")
for k, v in diffs_criterion.items():
    print(f"  {k}: {v:.4f}")

assert all(v <= 0 for v in diffs_treatment.values())
assert all(v <= 0 for v in diffs_criterion.values())
print()
print("PASSED in both scenarios -- the log-space engine is safe to use for the sweep.")


Treatment-scenario equivalence check (worst excess beyond atol+rtol*|raw|, as a
fraction of the allowed tolerance; <=0 means within tolerance):
  ns: 0.0000
  nr: 0.0000
  s: 0.0000
  a: 0.0000
  b: 0.0000



Criterion-scenario (a0=100, i=10) equivalence check:
  ns: 0.0000
  nr: 0.0000
  s: 0.0000
  a: 0.0000
  b: 0.0000

PASSED in both scenarios -- the log-space engine is safe to use for the sweep.


## Step 2 -- Choosing the integration horizon

Two things need pinning down before a grid sweep: how long to integrate (long enough
that essentially every grid point has reached steady state) and confirmation that the
steady-state detector itself is doing something sensible.

One subtlety already caught during development: a naive "relative rate of change"
detector on every state variable never converges, because once resistant cells are
gone the antibiotic simply decays as `a0 * exp(-d_a * t)`, which has a *constant*
relative rate (`d_a`) forever -- that is what exponential decay means, not a sign that
the system hasn't settled. `sweep_engine.time_to_steady_state()` instead gates only on
the two populations' own per-capita growth rates (`d ln n_s/dt`, `d ln n_r/dt`), which
are already the biologically meaningful quantities and do go to zero once growth and
killing have both stopped mattering.

In [3]:
ns0_ref = no_drug_reference()
print(f"Drug-free reference n_s (steady state, base strain alpha=0.95): {ns0_ref:.4f}")
print("(One fixed yardstick for 'S is viable' across the grid: with a=0 there is no lysis,")
print(" so beta_min, c and gamma do not matter; it is the base mixture's drug-free S density.)")


Drug-free reference n_s (steady state, base strain alpha=0.95): 2.3350
(One fixed yardstick for 'S is viable' across the grid: with a=0 there is no lysis,
 so beta_min, c and gamma do not matter; it is the base mixture's drug-free S density.)


In [4]:
corners = [(a, b, c) for a in ALPHA_RANGE for b in BETA_MIN_RANGE for c in C_RANGE]
corners.append((0.95, 0.9, 0.7))  # the paper's own Fig. 3A base case

print(f"{'alpha':>6} {'beta_min':>9} {'c':>5}  {'t_SS':>7} {'converged':>10}")
worst_t_ss = 0.0
for alpha, beta_min, c in corners:
    ep = compute_endpoints(alpha, beta_min, c, a0=2.15, i_dose=10.0,
                            ns0_no_drug=ns0_ref, t_span=(0, 150), n_points=300)
    worst_t_ss = max(worst_t_ss, ep.t_steady_state)
    print(f"{alpha:6.2f} {beta_min:9.2f} {c:5.2f}  {ep.t_steady_state:7.2f} {str(ep.ss_converged):>10}")
print(f"\nWorst time-to-steady-state among the cube's corners: {worst_t_ss:.2f}")


 alpha  beta_min     c     t_SS  converged
  0.75      0.00  0.00     7.02       True
  0.75      0.00  1.00    39.13       True
  0.75      1.00  0.00    12.54       True
  0.75      1.00  1.00    39.63       True
  1.00      0.00  0.00     5.52       True


  1.00      0.00  1.00    23.08       True


  1.00      1.00  0.00     9.53       True
  1.00      1.00  1.00    23.58       True
  0.95      0.90  0.70    15.55       True

Worst time-to-steady-state among the cube's corners: 39.63


In [5]:
rng = np.random.default_rng(7)
n_check = 60
alphas = rng.uniform(*ALPHA_RANGE, n_check)
beta_mins = rng.uniform(*BETA_MIN_RANGE, n_check)
cs = rng.uniform(*C_RANGE, n_check)

worst_t_ss_random = 0.0
n_unconverged = 0
for alpha, beta_min, c in zip(alphas, beta_mins, cs):
    ep = compute_endpoints(alpha, beta_min, c, a0=2.15, i_dose=10.0,
                            ns0_no_drug=ns0_ref, t_span=(0, 150), n_points=300)
    worst_t_ss_random = max(worst_t_ss_random, ep.t_steady_state)
    n_unconverged += (not ep.ss_converged)

print(f"Worst t_SS among {n_check} random interior points: {worst_t_ss_random:.2f}")
print(f"Unconverged count: {n_unconverged} / {n_check}")
print()
worst = max(worst_t_ss, worst_t_ss_random)
print(f"t_span=(0, 150) leaves {150 / worst:.1f}x headroom over the worst observed convergence")
print("time -- adopted as the horizon for the full treatment-scenario grid below.")
assert n_unconverged == 0 and worst < 150 / 2


Worst t_SS among 60 random interior points: 36.12
Unconverged count: 0 / 60

t_span=(0, 150) leaves 3.8x headroom over the worst observed convergence
time -- adopted as the horizon for the full treatment-scenario grid below.


## Step 3 -- Run the treatment-scenario grid (a0=2.15, i=10)

21 points per axis (0.75-1 for alpha, 0-1 for beta_min and c) = 9,261 combinations,
each a full ODE integration with all six endpoints extracted. Run in parallel across
both CPUs.

In [6]:
t0 = time.time()
treatment_df = run_treatment_grid(
    n_per_axis=21,
    alpha_range=ALPHA_RANGE, beta_min_range=BETA_MIN_RANGE, c_range=C_RANGE,
    a0=2.15, i_dose=10.0, t_span=(0.0, 150.0), n_points=300,
)
elapsed = time.time() - t0
print(f"Treatment grid: {len(treatment_df)} points in {elapsed:.1f}s "
      f"({elapsed / len(treatment_df) * 1000:.2f} ms/point)")
print(f"Unconverged points (t_SS hit the horizon without settling): "
      f"{(~treatment_df.ss_converged).sum()} / {len(treatment_df)}")


Treatment grid: 9261 points in 125.4s (13.54 ms/point)
Unconverged points (t_SS hit the horizon without settling): 0 / 9261


In [7]:
treatment_df.to_csv("treatment_grid.csv", index=False)
print("Saved treatment_grid.csv")
treatment_df.describe()


Saved treatment_grid.csv


,alpha,beta_min,c,f_R_final,t_recovery_S,t_steady_state,peak_loss_S,peak_loss_total,diversity_final,diversity_min,ns_final,nr_final
count,9261.000000,9261.000000,9261.000000,9261.000000,9261.000000,9261.000000,9261.000000,9261.000000,9261.000000,9261.000000,9261.000000,9261.000000
mean,0.875000,0.500000,0.500000,0.479791,12.527288,14.382083,0.338029,0.268361,0.560228,0.549447,1.969425,1.942252
std,0.075695,0.302781,0.302781,0.240190,7.231764,7.569380,0.140012,0.248284,0.166073,0.162459,0.766586,1.025873
min,0.750000,0.000000,0.000000,0.000912,4.013378,5.518395,0.141097,0.000000,0.007294,0.007294,0.672791,0.002795
25%,0.812500,0.250000,0.250000,0.300247,7.525084,9.030100,0.223254,0.022828,0.523975,0.511402,1.299489,1.146818
50%,0.875000,0.500000,0.500000,0.528735,10.033445,11.538462,0.298816,0.217344,0.620259,0.601819,1.870337,2.107684
75%,0.937500,0.750000,0.750000,0.683422,15.551839,17.056856,0.431234,0.454768,0.675413,0.663117,2.666736,2.811373
max,1.000000,1.000000,1.000000,0.844901,36.622074,39.632107,0.708084,0.843546,0.693147,0.693147,3.545330,3.665014


## Step 4 -- Run the criterion-validation grid (a0=100, i=10, highest doses in the paper's range)

41 points per axis = 68,921 combinations (about 7x the paper's 10,000 random strains),
over the same SI ranges. Here we only need the criterion's binary prediction and the
simulated outcome at each point, so the grid can be finer than the treatment grid.

**Horizon check first.** With d_a = 0.02, an antibiotic dose of a0 = 100 takes a few
hundred time units to decay, and the populations then regrow. The simulated outcome
(sign of the change in ln(n_r/n_s) over the whole run) is compared at t = 600 and
t = 1500 on a random sample to confirm 600 is long enough.

In [8]:
rng = np.random.default_rng(11)
flips = 0
n_hcheck = 200
for _ in range(n_hcheck):
    a, bm, cc = (rng.uniform(*ALPHA_RANGE), rng.uniform(*BETA_MIN_RANGE), rng.uniform(*C_RANGE))
    r600 = criterion_check(a, bm, cc, a0=100.0, i_dose=10.0, t_span=(0.0, 600.0))
    r1500 = criterion_check(a, bm, cc, a0=100.0, i_dose=10.0, t_span=(0.0, 1500.0))
    flips += r600["simulated_R_favored"] != r1500["simulated_R_favored"]
print(f"Simulated outcome changes between t=600 and t=1500: {flips} / {n_hcheck} random points")
assert flips == 0


Simulated outcome changes between t=600 and t=1500: 0 / 200 random points


In [9]:
t0 = time.time()
criterion_df = run_criterion_grid(
    n_per_axis=41,
    alpha_range=ALPHA_RANGE, beta_min_range=BETA_MIN_RANGE, c_range=C_RANGE,
    a0=100.0, i_dose=10.0, t_span=(0.0, 600.0), n_points=50,
)
elapsed = time.time() - t0
print(f"Criterion grid: {len(criterion_df)} points in {elapsed:.1f}s "
      f"({elapsed / len(criterion_df) * 1000:.2f} ms/point)")

agreement = (criterion_df.criterion_predicts_R == criterion_df.simulated_R_favored).mean()
false_neg = ((criterion_df.criterion_predicts_R) & (~criterion_df.simulated_R_favored)).sum()
false_pos = ((~criterion_df.criterion_predicts_R) & (criterion_df.simulated_R_favored)).sum()
print(f"\nAgreement: {agreement:.1%}")
print(f"  criterion says R-favored, simulation says S-favored: {false_neg}")
print(f"  criterion says S-favored, simulation says R-favored: {false_pos}")


Criterion grid: 68921 points in 1106.6s (16.06 ms/point)

Agreement: 93.5%
  criterion says R-favored, simulation says S-favored: 4396
  criterion says S-favored, simulation says R-favored: 55


In [10]:
criterion_df.to_csv("criterion_grid.csv", index=False)
print("Saved criterion_grid.csv")
criterion_df.head()


Saved criterion_grid.csv


,alpha,beta_min,c,x_val,y_val,criterion_predicts_R,simulated_R_favored,advantage
0,0.75,0.0,0.000,0.181159,1.000,True,True,3.723697
1,0.75,0.0,0.025,0.181159,0.975,True,True,3.743953
2,0.75,0.0,0.050,0.181159,0.950,True,True,3.765717
3,0.75,0.0,0.075,0.181159,0.925,True,True,3.789198
4,0.75,0.0,0.100,0.181159,0.900,True,True,3.814646


## Summary

- `treatment_grid.csv` -- 9,261 rows, one per (alpha, beta_min, c), with all six
  requested endpoints plus f_R, extinction/survival flags, and diversity. Analyzed in
  **02_endpoint_analysis.ipynb**.
- `criterion_grid.csv` -- 68,921 rows, the criterion's prediction vs. the simulated
  outcome at each point. Analyzed in **03_criterion_validation.ipynb**.

Both grids use the SI base values for every parameter that is not swept (gamma = 1.38,
h_a = 3, h_i = 2, kappa_b = 0.35, d_a = 0.02, d_b = 1, xi = 0.8, phi_max = 1) and the SI
initial conditions (n_s = n_r = 0.2, s = 4, b = 0). alpha, beta_min and c span the SI
randomized ranges.